## English to Yorùbá translator
Run the cells in order (Kernel > Restart and Run All).

In [ ]:
%run main_assignment.ipynb

### 1. Analysis

In [ ]:
import re

# The lexicon cells must already have been run (cell above does this with %run).
assert "LEXICON" in globals(), "Run main_assignment first (the cell above)."

# Rebuild the multiword list from LEXICON, so it is always in sync
MULTIWORD_KEYS = sorted((k for k in LEXICON if " " in k), key=len, reverse=True)

PUNCT = {"?", ".", "!", ",", ";"}

# ---------- Preprocessing ----------

CONTRACTIONS = {
    "isn't": ["is", "not"], "aren't": ["are", "not"], "wasn't": ["was", "not"],
    "weren't": ["were", "not"], "doesn't": ["does", "not"], "don't": ["do", "not"],
    "didn't": ["did", "not"], "won't": ["will", "not"], "hasn't": ["has", "not"],
    "haven't": ["have", "not"], "hadn't": ["had", "not"],
    "i'm": ["I", "am"], "you're": ["you", "are"], "we're": ["we", "are"],
    "they're": ["they", "are"], "he's": ["he", "is"], "she's": ["she", "is"],
    "it's": ["it", "is"], "i'll": ["I", "will"], "you'll": ["you", "will"],
    "he'll": ["he", "will"], "she'll": ["she", "will"], "we'll": ["we", "will"],
    "they'll": ["they", "will"], "i've": ["I", "have"], "you've": ["you", "have"],
    "we've": ["we", "have"], "they've": ["they", "have"],
}


def tokenize(sentence):
    sentence = sentence.replace("\u2019", "'").strip()
    return re.findall(r"\w+(?:'\w+)*|[?.!,;]", sentence)


def expand_contractions(words):
    out = []
    for w in words:
        lw = w.lower()
        if lw in CONTRACTIONS:
            out.extend(CONTRACTIONS[lw])
        elif lw.endswith("n't"):
            out.extend([w[:-3], "not"])
        elif lw.endswith("'s"):
            out.extend([w[:-2], "'s"])  # John's -> John 's (possessive)
        else:
            out.append(w)
    return out


# ---------- Word forms ----------

BE_FORMS = {"am", "is", "are", "was", "were", "be", "been"}
HAVE_FORMS = {"has", "have", "had"}
DO_FORMS = {"do", "does", "did"}
OBJ_PRONOUNS = {"me": "i", "him": "he", "them": "they", "us": "we"}
# Yorùbá ó is the same for he / she / it, but we keep the English gender here
GENDER = {"he": "masc", "she": "fem", "it": "neut"}

IRREGULAR_VERBS = {
    "ate": ("eat", "PAST"), "eaten": ("eat", "PART"),
    "went": ("go", "PAST"), "gone": ("go", "PART"),
    "came": ("come", "PAST"),
    "saw": ("see", "PAST"), "seen": ("see", "PART"),
    "heard": ("hear", "PAST"), "said": ("say", "PAST"),
    "wrote": ("write", "PAST"), "written": ("write", "PART"),
    "slept": ("sleep", "PAST"), "stood": ("stand", "PAST"), "sat": ("sit", "PAST"),
    "bought": ("buy", "PAST"), "sold": ("sell", "PAST"),
    "gave": ("give", "PAST"), "given": ("give", "PART"),
    "did": ("do", "PAST"), "done": ("do", "PART"),
    "knew": ("know", "PAST"), "known": ("know", "PART"),
    "taught": ("teach", "PAST"),
    "drank": ("drink", "PAST"), "drunk": ("drink", "PART"),
}
IRREGULAR_PLURALS = {"children": "child", "people": "person"}


def is_lexicon(word, pos):
    return word in LEXICON and LEXICON[word]["pos"] == pos


def verb_form(word):
    """Return (base, form) if word is a form of a lexicon verb, else None.
    form is one of BASE, ING, PAST, PART, 3SG."""
    w = word.lower()
    if is_lexicon(w, "V"):
        return (w, "BASE")
    if w in IRREGULAR_VERBS and is_lexicon(IRREGULAR_VERBS[w][0], "V"):
        return IRREGULAR_VERBS[w]
    cands = []
    if w.endswith("ing") and len(w) > 4:
        stem = w[:-3]
        cands += [(stem, "ING"), (stem + "e", "ING")]
        if len(stem) > 1 and stem[-1] == stem[-2]:
            cands.append((stem[:-1], "ING"))
    if w.endswith("ed") and len(w) > 3:
        stem = w[:-2]
        cands += [(stem, "PAST"), (w[:-1], "PAST")]
        if len(stem) > 1 and stem[-1] == stem[-2]:
            cands.append((stem[:-1], "PAST"))
        if w.endswith("ied"):
            cands.append((w[:-3] + "y", "PAST"))
    if w.endswith("ies"):
        cands.append((w[:-3] + "y", "3SG"))
    if w.endswith("es"):
        cands.append((w[:-2], "3SG"))
    if w.endswith("s"):
        cands.append((w[:-1], "3SG"))
    for base, form in cands:
        if is_lexicon(base, "V"):
            return (base, form)
    return None


def noun_plural(word):
    """Return the singular if word is a plural of a lexicon noun, else None."""
    w = word.lower()
    if w in IRREGULAR_PLURALS:
        return IRREGULAR_PLURALS[w]
    cands = []
    if w.endswith("ies"):
        cands.append(w[:-3] + "y")
    if w.endswith("es"):
        cands.append(w[:-2])
    if w.endswith("s"):
        cands.append(w[:-1])
    for base in cands:
        if is_lexicon(base, "N"):
            return base
    return None


def next_content_index(words, i):
    """Index of the next word after i, skipping 'not'."""
    while i < len(words) and words[i].lower() == "not":
        i += 1
    return i


# ---------- Lookup ----------

def lookup(word, pending_tense=None):
    lw = word.lower()

    if lw in OBJ_PRONOUNS:
        base = OBJ_PRONOUNS[lw]
        tok = {"en": word, "base": base, "role": "obj", **LEXICON[base]}
        if base in GENDER:
            tok["gender"] = GENDER[base]
        return tok

    if lw == "'s":
        return {"en": word, "pos": "POSS_S", "yo": None,
                "note": "John's book -> ìwé John (owner goes after the thing)"}

    if lw in LEXICON:
        tok = {"en": word, "base": lw, **LEXICON[lw]}
        if lw in GENDER:
            tok["gender"] = GENDER[lw]
        if tok["pos"] == "V":
            tok["form"] = "BASE"
            if pending_tense:
                tok["tense"] = pending_tense
        return tok

    vf = verb_form(lw)
    if vf:
        base, form = vf
        tok = {"en": word, "base": base, "pos": "V",
               "yo": LEXICON[base]["yo"], "form": form}
        if form == "PAST" or pending_tense == "PAST":
            tok["tense"] = "PAST"
        return tok

    singular = noun_plural(lw)
    if singular:
        return {"en": word, "base": singular, "pos": "N",
                "yo": LEXICON[singular]["yo"], "number": "PL"}

    if word[0].isupper():
        return {"en": word, "pos": "PN", "yo": word}  # proper noun, passed through
    return {"en": word, "pos": "UNK", "yo": None, "note": "not in lexicon"}


# ---------- Analysis ----------

def analyze(sentence):
    words = expand_contractions(tokenize(sentence))
    n = len(words)
    tokens = []
    pending_tense = None
    i = 0

    while i < n:
        w = words[i]
        lw = w.lower()

        if lw in PUNCT:
            tokens.append({"en": w, "pos": "PUNCT", "yo": None})
            i += 1
            continue

        # Multiword entries first ("pounded yam", "you all")
        matched = False
        for key in MULTIWORD_KEYS:
            parts = key.split()
            if [x.lower() for x in words[i:i + len(parts)]] == parts:
                tokens.append({"en": " ".join(words[i:i + len(parts)]), "base": key,
                               **LEXICON[key]})
                i += len(parts)
                matched = True
                break
        if matched:
            continue

        # Auxiliaries become aspect markers (or are dropped)
        if lw == "will":
            tokens.append({"en": w, "pos": "ASP", "tag": "FUT", "yo": None})
            i += 1
            continue

        if lw in BE_FORMS:
            j = next_content_index(words, i + 1)
            vf = verb_form(words[j]) if j < n else None
            if vf and vf[1] == "ING":
                tokens.append({"en": w, "pos": "ASP", "tag": "PROG", "yo": None,
                               "tense": "PAST" if lw in ("was", "were") else "PRES"})
            else:
                tokens.append({"en": w, "pos": "COP", "yo": None,
                               "note": "be as main verb; the rules decide ni / jẹ́ / wà"})
            i += 1
            continue

        if lw in HAVE_FORMS:
            j = next_content_index(words, i + 1)
            vf = verb_form(words[j]) if j < n else None
            if vf and vf[1] in ("PAST", "PART"):
                tokens.append({"en": w, "pos": "ASP", "tag": "PERF", "yo": None})
                i += 1
                continue

        if lw in DO_FORMS:
            nxt = words[i + 1].lower() if i + 1 < n else ""
            if nxt == "not" or (i == 0 and words[-1] == "?"):
                if lw == "did":
                    pending_tense = "PAST"
                i += 1  # do-support has no Yorùbá equivalent, so drop it
                continue

        tokens.append(lookup(w, pending_tense))
        if tokens[-1]["pos"] == "V":
            pending_tense = None
        i += 1

    # Pronoun role: before the verb = subject, after = object.
    # A new clause (and / but / because / who / comma) starts the count again.
    seen_verb = False
    for tok in tokens:
        if tok["pos"] in ("CONJ", "SUB", "REL") or (
                tok["pos"] == "PUNCT" and tok["en"] == ","):
            seen_verb = False
        if tok["pos"] == "V":
            seen_verb = True
        if tok["pos"] == "PRON":
            role = tok.get("role") or ("obj" if seen_verb else "subj")
            tok["role"] = role
            tok["yo"] = tok[role]
            if tok["yo"] is None:
                tok["note"] = "3sg object copies the verb's final vowel"

    return tokens


def show(tokens):
    parts = []
    for t in tokens:
        label = t["pos"] + ("." + t["tag"] if "tag" in t else "")
        if "form" in t and t["form"] != "BASE":
            label += "." + t["form"]
        if t.get("number"):
            label += "." + t["number"]
        if t.get("gender"):
            label += "." + t["gender"]
        parts.append(f"{t['en']}[{label}]" + (f"={t['yo']}" if t.get("yo") else ""))
    return "  ".join(parts)


### 2. Extra words (or, who, when...) and sentence types

In [ ]:
import unicodedata

# ---------- Extra words for compound and complex sentences ----------
# Added here so main_assignment stays untouched. REVIEW every Yorùbá form.
EXTRA_ENTRIES = [
    ("or", "CONJ", "tàbí"),
    ("that", "REL", "tí"),
    ("who", "REL", "tí"),
    ("which", "REL", "tí"),
    ("when", "SUB", "nígbà tí"),
    ("if", "SUB", "bí"),
    ("although", "SUB", "bí ó tilẹ̀ jẹ́ pé"),
    ("after", "SUB", "lẹ́yìn tí"),
]
for english, pos, yoruba in EXTRA_ENTRIES:
    LEXICON.setdefault(english, {"pos": pos, "yo": unicodedata.normalize("NFC", yoruba)})

# "because" starts a dependent clause, so it is tagged SUB, not CONJ
LEXICON["because"]["pos"] = "SUB"

# Tags: CONJ = and / but / or (joins equal clauses)
#       SUB  = because / when / if (starts a dependent clause)
#       REL  = who / which / that (relative clause)

SUBJECT_LIKE = {"PRON", "PN", "N", "DET", "POSS"}
VERBAL = {"V", "COP", "ASP"}


def starts_clause(tokens, i):
    """True if the CONJ at position i is followed by a new subject + verb
    (so it joins two clauses, not just two nouns)."""
    seg = []
    j = i + 1
    while j < len(tokens) and tokens[j]["pos"] != "CONJ" and not (
            tokens[j]["pos"] == "PUNCT" and tokens[j]["en"] in ".?!"):
        seg.append(tokens[j]["pos"])
        j += 1
    for k, pos in enumerate(seg):
        if pos in SUBJECT_LIKE and any(p in VERBAL for p in seg[k + 1:]):
            return True
    return False


def clause_info(tokens):
    """Classify a tagged sentence as simple, compound, complex or compound-complex."""
    # Dependent clauses = subordinators + relative markers
    n_dep = sum(1 for t in tokens if t["pos"] in ("SUB", "REL"))

    # Each CONJ that starts a new subject + verb adds an independent clause
    n_new = sum(1 for i, t in enumerate(tokens)
                if t["pos"] == "CONJ" and starts_clause(tokens, i))

    n_indep = 1 + n_new
    if n_indep >= 2 and n_dep >= 1:
        kind = "compound-complex"
    elif n_indep >= 2:
        kind = "compound"
    elif n_dep >= 1:
        kind = "complex"
    else:
        kind = "simple"
    return {"type": kind, "independent": n_indep, "dependent": n_dep}


def describe(sentence):
    tokens = analyze(sentence)
    info = clause_info(tokens)
    print(sentence)
    print(f"   Type: {info['type']} "
          f"({info['independent']} independent, {info['dependent']} dependent)")
    print("  ", show(tokens), "\n")


### 3. Transfer rules

In [ ]:
# ---------- Transfer rules ----------
# Each rule takes a list of tagged tokens and returns a new list.


# Habitual markers (simple present of an action verb). REVIEW the negative form.
MARKERS.setdefault("HAB", "máa ń")      # he eats fish -> ó máa ń jẹ ẹja
MARKERS.setdefault("HAB_NEG", "kí ń")   # I don't eat rice -> mi ò kí ń jẹ ìrẹsì

# Verbs that describe a state, so they do not take máa ń
STATIVE_VERBS = {"want", "love", "know", "see", "hear", "need"}

# is / are + adjective: Yorùbá uses a stative verb (ó tóbi). REVIEW and extend.
PREDICATE_ADJ = {"big": "tóbi", "small": "kéré", "good": "dára", "bad": "burú",
                 "red": "pupa", "white": "funfun", "black": "dúdú",
                 "old": "ti gbó"}  # REVIEW: gbó = aged / worn with age; ti assumed


def mark_habitual(tokens):
    """Simple present of an action verb is habitual, so add a HAB marker before it."""
    out, has_subject = [], False
    for t in tokens:
        if t["pos"] in ("CONJ", "SUB") or (t["pos"] == "PUNCT" and t["en"] == ","):
            has_subject = False
        elif t["pos"] in ("REL", "PRON", "PN", "N"):
            has_subject = True
        if (t["pos"] == "V" and t.get("form") in ("BASE", "3SG")
                and t.get("tense") != "PAST" and t.get("base") not in STATIVE_VERBS
                and has_subject):
            prev = next((x for x in reversed(out) if x["pos"] != "NEG"), None)
            blocked = prev and (prev["pos"] == "ASP" or
                                (prev["pos"] == "PREP" and prev["en"].lower() == "to"))
            if not blocked:
                out.append({"en": "(habitual)", "pos": "ASP", "tag": "HAB", "yo": None})
        out.append(t)
    return out


def apply_predicate_adj(tokens):
    """is / are + adjective -> a stative verb:  ó tóbi (not 'ó jẹ́ ńlá')."""
    out, drop = list(tokens), set()
    for i, t in enumerate(out):
        if t["pos"] == "COP":
            j = i + 1
            while j < len(out) and out[j]["pos"] == "NEG":
                j += 1
            if j < len(out) and out[j]["pos"] == "ADJ":
                form = PREDICATE_ADJ.get(out[j].get("base"))
                if form:
                    out[j] = {**out[j], "yo": form, "pos": "V"}
                    drop.add(i)
    return [t for k, t in enumerate(out) if k not in drop]


def apply_aspect(tokens):
    """Fill in the aspect markers (ń, ti, máa, máa ń)."""
    out = []
    for i, t in enumerate(tokens):
        if t["pos"] == "ASP":
            tag = t["tag"]
            if tag == "HAB" and i > 0 and tokens[i - 1]["pos"] == "NEG":
                tag = "HAB_NEG"
            out.append({**t, "yo": MARKERS[tag]})
        else:
            out.append(t)
    return out


NEG_MERGE = {"mo": "mi ò", "o": "o ò", "a": "a ò", "ẹ": "ẹ ò", "wọ́n": "wọn ò", "ó": "kò"}


def apply_negation(tokens):
    """Negative goes before the aspect marker; subject pronoun merges with it."""
    toks = [dict(t) for t in tokens]
    for i in range(1, len(toks)):
        if toks[i]["pos"] == "NEG" and toks[i - 1]["pos"] == "ASP":
            toks[i - 1], toks[i] = toks[i], toks[i - 1]
            if toks[i]["tag"] == "FUT":
                toks[i]["yo"] = "ní"
    out = []
    for t in toks:
        prev = out[-1] if out else None
        if (t["pos"] == "NEG" and prev and prev["pos"] == "PRON"
                and prev.get("role") == "subj" and prev["yo"] in NEG_MERGE):
            prev["yo"] = NEG_MERGE[prev["yo"]]
            continue
        out.append(t)
    return out


def apply_possessive_s(tokens):
    """John's book -> book John (the owner goes after the thing)."""
    out = list(tokens)
    i = 1
    while i < len(out) - 1:
        if (out[i]["pos"] == "POSS_S" and out[i - 1]["pos"] in ("PN", "N")
                and out[i + 1]["pos"] == "N"):
            out[i - 1:i + 2] = [out[i + 1], out[i - 1]]
            i = 1
        else:
            i += 1
    return out


NP_TYPES = {"DET", "POSS", "ADJ", "N"}


def apply_noun_phrases(tokens):
    """English: DET/POSS ADJ N.  Yorùbá: (àwọn) N ADJ POSS DET."""
    out, i = [], 0
    while i < len(tokens):
        if tokens[i]["pos"] not in NP_TYPES:
            out.append(tokens[i])
            i += 1
            continue

        # Collect one noun phrase (a new DET/POSS after a noun starts a new phrase)
        j, seen_n = i, False
        while j < len(tokens) and tokens[j]["pos"] in NP_TYPES:
            if tokens[j]["pos"] in ("DET", "POSS") and seen_n:
                break
            if tokens[j]["pos"] == "N":
                seen_n = True
            j += 1
        span = tokens[i:j]

        heads = [t for t in span if t["pos"] == "N"]
        if not heads:
            out.extend(span)  # e.g. a lone adjective ("are old")
        else:
            head = heads[0]
            after_prep = i > 0 and tokens[i - 1]["pos"] == "PREP"
            if head.get("number") == "PL":
                out.append({"en": "(plural)", "pos": "PL", "yo": "àwọn"})
            out.append(head)
            out.extend(t for t in span if t["pos"] == "ADJ")
            out.extend(t for t in span if t["pos"] == "POSS")
            for t in span:
                if t["pos"] == "DET":
                    if t["en"].lower() == "the" and after_prep:
                        continue  # "to the market" -> sí ọjà
                    out.append(t)
        i = j
    return out


def apply_verb_forms(tokens):
    """be as a main verb -> jẹ́; kọ́ni with an object -> kọ́."""
    out = [dict(t) for t in tokens]
    for i, t in enumerate(out):
        if t["pos"] == "COP":
            t["yo"] = "jẹ́"
        if (t["pos"] == "V" and t["yo"] == "kọ́ni" and i + 1 < len(out)
                and out[i + 1]["pos"] in ("N", "PL", "PRON", "PN")):
            t["yo"] = "kọ́"
    return out


def apply_relative(tokens):
    """who / which / that -> tí. When the clause starts with a verb, tí is followed
    by a pronoun: ọmọ tí ó jẹ ìrẹsì (ó for one, wọ́n for plural). With kò there is
    no extra pronoun: ọmọ tí kò jẹ ìrẹsì."""
    out = []
    for i, t in enumerate(tokens):
        out.append(t)
        if t["pos"] == "REL":
            nxt = tokens[i + 1] if i + 1 < len(tokens) else None
            if nxt and nxt["pos"] in ("V", "ASP", "COP"):
                head = next((x for x in reversed(out[:-1]) if x["pos"] == "N"), None)
                pron = "wọ́n" if head and head.get("number") == "PL" else "ó"
                out.append({"en": "(pronoun)", "pos": "PRON", "yo": pron})
    return out


def mark_clause_conj(tokens):
    """Flag each and / but / or that joins two clauses (not just two nouns)."""
    out = [dict(t) for t in tokens]
    for i, t in enumerate(out):
        if t["pos"] == "CONJ":
            t["joins_clauses"] = starts_clause(tokens, i)
    return out


def apply_conjunctions(tokens):
    """Join clauses the Yorùbá way.
    and -> sì, placed after the new subject:  Mo jẹ ìrẹsì, ó sì mu omi
    but / or -> ṣùgbọ́n / tàbí after a comma.
    and between two nouns stays àti."""
    out, pending_si = [], False
    for t in tokens:
        if t["pos"] == "CONJ" and t.get("joins_clauses"):
            if not (out and out[-1]["pos"] == "PUNCT" and out[-1]["en"] == ","):
                out.append({"en": ",", "pos": "PUNCT", "yo": None})
            if t["en"].lower() == "and":
                pending_si = True  # the word itself is dropped; sì is added below
                continue
            out.append(t)
            continue
        if pending_si and t["pos"] in ("V", "COP", "ASP", "NEG"):
            out.append({"en": "and", "pos": "CONJ", "yo": "sì"})
            pending_si = False
        out.append(t)
    return out


def apply_clause_order(tokens):
    """'When he came, I was eating rice' -> 'Mo ń jẹ ìrẹsì, nígbà tí ó wá'."""
    if not tokens or tokens[0]["pos"] != "SUB" or tokens[0]["en"].lower() != "when":
        return tokens
    comma = next((i for i, t in enumerate(tokens)
                  if t["pos"] == "PUNCT" and t["en"] == ","), None)
    if comma is None:
        return tokens
    sub, main = tokens[:comma], tokens[comma + 1:]
    end = []
    if main and main[-1]["pos"] == "PUNCT" and main[-1]["en"] in ".?!":
        end, main = [main[-1]], main[:-1]
    return main + [{"en": ",", "pos": "PUNCT", "yo": None}] + sub + end


def apply_question(tokens):
    """A question gets ṣé at the start."""
    if any(t["pos"] == "PUNCT" and t["en"] == "?" for t in tokens):
        return [{"en": "?", "pos": "QM", "yo": MARKERS["QM"]}] + tokens
    return tokens


def render(tokens):
    text = ""
    for t in tokens:
        if t["pos"] == "PUNCT":
            text += t["en"]
        elif t.get("yo"):
            text += (" " if text else "") + t["yo"]
    return text[:1].upper() + text[1:]


def translate(sentence):
    tokens = analyze(sentence)
    for rule in (mark_clause_conj, mark_habitual, apply_predicate_adj, apply_aspect,
                 apply_negation, apply_possessive_s, apply_noun_phrases,
                 apply_verb_forms, apply_relative, apply_conjunctions,
                 apply_clause_order, apply_question):
        tokens = rule(tokens)
    return render(tokens)


### 4. Test

In [ ]:
groups = {
    "SIMPLE": [
        "He is eating rice",
        "I don't want water",
        "My father bought a big car",
        "Adé will not go to the market",
        "The big houses are old",
        "Did you see the child?",
        "They have gone to the market",
        "She teaches my children",
        "John's book",
        "I eat pounded yam",
    ],
    "COMPOUND": [
        "I eat rice and he drinks water",
        "My father bought a car but my mother bought a book",
        "She will go to the market or he will go to school",
        "I don't eat rice but he eats fish",
        "I eat rice and fish",
    ],
    "COMPLEX": [
        "The child who ate rice is big",
        "She went to the market because she wanted bread",
        "When he came, I was eating rice",
        "The rice that he ate was big",
        "The child who did not eat rice is big",
    ],
    "COMPOUND-COMPLEX": [
        "My father bought a car because he wanted a big house, but my mother bought a book",
        "The teacher who taught my child went to the market and she bought fish because she wanted food",
        "When he came, I was eating rice but she was sleeping",
    ],
    "EXTRA CHECKS": [
        "Do you eat rice?",
        "The house is red",
        "He is not big",
        "The children who eat rice are small",
    ],
}
for name, sentences in groups.items():
    print(name)
    for s in sentences:
        print(s, "->", translate(s))
    print()

# Gender is kept in the analysis even though ó is the same for he / she / it
print(show(analyze("He eats rice")))
print(show(analyze("She eats rice")))